In [2]:
!pip install scikit-surprise


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.4/154.4 kB 2.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for scikit-surprise: filename=scikit_surprise-1.1.4-cp311-cp311-linux_x86_64.whl size=2505166 sha256=6fd7c6a3b99df87da3ea773c6d16d676b85e95926d1e211edd223f28bce916b5
  Stored in directory: /root/.cache/pip/wheels/2a/8f/6e/7e2899163e2d85d8266daab4aa1cdabec7a6c56f83c015b5af
Successfully built scikit-surprise


In [ ]:
import pandas as pd
import numpy as np
from surprise import Dataset, Reader, SVDpp, KNNBasic
from surprise.model_selection import train_test_split, GridSearchCV
from collections import defaultdict
from sklearn.metrics import precision_score, recall_score, f1_score

# Load the MovieLens 20M dataset
import kagglehub
path = kagglehub.dataset_download("grouplens/movielens-20m-dataset")

# Load ratings and movies data
ratings_df = pd.read_csv(f"{path}/rating.csv")
movies_df = pd.read_csv(f"{path}/movie.csv")

# Handle sparsity by filling missing values with user mean ratings
user_mean_ratings = ratings_df.groupby('userId')['rating'].transform('mean')
ratings_df['rating_filled'] = ratings_df['rating'].fillna(user_mean_ratings)

# Prepare the data for Surprise
reader = Reader(rating_scale=(0.5, 5))
data = Dataset.load_from_df(ratings_df[['userId', 'movieId', 'rating_filled']], reader)

# Train-test split (using 75% of the data for training)
trainset, testset = train_test_split(data, test_size=0.25)
print(f"Training data size: {trainset.n_ratings}, Test data size: {len(testset)}")

# Optimized SVD++ model
svdpp_model = SVDpp()
svdpp_model.fit(trainset)

# KNN-based Collaborative Filtering (User-Based CF)
sim_options_user = {'name': 'cosine', 'user_based': True}
knn_user_model = KNNBasic(sim_options=sim_options_user)
knn_user_model.fit(trainset)

# KNN-based Collaborative Filtering (Item-Based CF)
sim_options_item = {'name': 'cosine', 'user_based': False}
knn_item_model = KNNBasic(sim_options=sim_options_item)
knn_item_model.fit(trainset)

# Function to get top-N recommendations
def get_top_n(predictions, n=5):
    top_n = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        top_n[uid].append((iid, est))
    for uid, user_ratings in top_n.items():
        user_ratings.sort(key=lambda x: x[1], reverse=True)
        top_n[uid] = user_ratings[:n]
    return top_n

# Make predictions
svdpp_predictions = svdpp_model.test(testset)
knn_user_predictions = knn_user_model.test(testset)
knn_item_predictions = knn_item_model.test(testset)

# Get recommendations
svdpp_top_n = get_top_n(svdpp_predictions, n=5)
knn_user_top_n = get_top_n(knn_user_predictions, n=5)
knn_item_top_n = get_top_n(knn_item_predictions, n=5)

# Hybrid Model - Combining SVD++ and KNN predictions
def hybrid_recommendations(user_id, n=5):
    svdpp_scores = {iid: score for iid, score in svdpp_top_n.get(user_id, [])}
    knn_scores = {iid: score for iid, score in knn_user_top_n.get(user_id, [])}

    combined_scores = {iid: (svdpp_scores.get(iid, 0) + knn_scores.get(iid, 0)) / 2 for iid in set(svdpp_scores) | set(knn_scores)}

    top_items = sorted(combined_scores.items(), key=lambda x: x[1], reverse=True)[:n]
    return top_items

# Display sample recommendations
sample_user = list(svdpp_top_n.keys())[0]
print(f"\nHybrid Recommendations for User {sample_user}:")
for movie_id, rating in hybrid_recommendations(sample_user, n=5):
    movie_title = movies_df[movies_df['movieId'] == movie_id]['title'].values[0]
    print(f"- {movie_title} (Predicted Rating: {rating:.2f})")

# Evaluation
y_true = [true_r for (_, _, true_r, _) in svdpp_predictions]
y_pred = [est for (_, _, _, est) in svdpp_predictions]

precision = precision_score(y_true, y_pred, average='micro')
recall = recall_score(y_true, y_pred, average='micro')
f1 = f1_score(y_true, y_pred, average='micro')

print("\nEvaluation Metrics:")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
